# Network-based bicyclist crash clusters

This notebook looks for groups of severe or fatal bicyclist crashes that are close **along the street network**, rather than merely close as the crow flies. It is intentionally simple: it uses network distance plus DBSCAN and saves a cluster table for reporting.

This identifies concentrations, not crash risk. A risk rate would require bicyclist or traffic exposure data.

## 1. Install once

From the repository folder, run:

```bash
python -m pip install -r requirements.txt
```

In [13]:
from pathlib import Path
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import osmnx as ox
import networkx as nx
from sklearn.cluster import DBSCAN

ROOT = Path.cwd()
RAW = ROOT / 'data' / 'raw'
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')


## 2. Load the CRIS export

The export has 12 metadata rows before its header. We keep one row per crash after filtering to San Antonio pedalcyclists with a fatal or suspected serious injury.

In [14]:
raw = pd.read_csv(RAW / 'myexport_final.csv', skiprows=12, low_memory=False)
severity = {'K - FATAL INJURY', 'A - SUSPECTED SERIOUS INJURY'}
people = raw[(raw['City'] == 'SAN ANTONIO') &
             (raw['Person Type'] == '3 - PEDALCYCLIST') &
             (raw['Person Injury Severity'].isin(severity))].copy()
people['year'] = pd.to_numeric(people['Crash Year'], errors='coerce')
people['latitude'] = pd.to_numeric(people['Latitude'], errors='coerce')
people['longitude'] = pd.to_numeric(people['Longitude'], errors='coerce')
people['death'] = (people['Person Injury Severity'] == 'K - FATAL INJURY').astype(int)
people['serious'] = (people['Person Injury Severity'] == 'A - SUSPECTED SERIOUS INJURY').astype(int)
crashes = (people.groupby('Crash ID', as_index=False)
                 .agg(year=('year', 'first'), latitude=('latitude', 'first'),
                      longitude=('longitude', 'first'), deaths=('death', 'sum'),
                      serious_injuries=('serious', 'sum')))
crashes = crashes.dropna(subset=['latitude', 'longitude']).copy()
crashes = crashes[crashes['year'].between(2016, 2025)].copy()
print(f'{len(crashes)} geocoded crashes loaded')
crashes.head()


250 geocoded crashes loaded


,Crash ID,year,latitude,longitude,deaths,serious_injuries
0,14852242,2016,29.430195,-98.464042,0,1
1,14863653,2016,29.465439,-98.565852,0,1
2,14864296,2016,29.404383,-98.511041,0,1
3,14874540,2016,29.412756,-98.605691,1,1
5,14947145,2016,29.474485,-98.518042,0,1


## 3. Download the San Antonio street network

This uses OpenStreetMap's drivable street network as the analysis network. The goal is not to measure driver fault; it is to measure whether crash locations are close along connected streets.

In [15]:
G = ox.graph_from_place('San Antonio, Texas, USA', network_type='drive', simplify=True)
G = ox.add_edge_lengths(G)
nodes = ox.distance.nearest_nodes(G, X=crashes['longitude'].tolist(), Y=crashes['latitude'].tolist())
crashes['network_node'] = nodes
print(f'{len(G.nodes):,} network nodes and {len(G.edges):,} street edges')


AttributeError: module 'osmnx' has no attribute 'add_edge_lengths'

## 4. Find clusters

`eps` is the maximum distance along the network between crashes in a cluster. We test 250, 500 and 1,000 meters. `min_samples=3` means a cluster needs at least three crashes. DBSCAN labels isolated crashes as `-1`.

In [ ]:
unique_nodes = crashes['network_node'].drop_duplicates().tolist()
distances = {}
for node in unique_nodes:
    distances[node] = nx.single_source_dijkstra_path_length(G, node, cutoff=1000, weight='length')

def network_matrix(node_series):
    matrix = []
    for a in node_series:
        row = []
        for b in node_series:
            row.append(distances.get(a, {}).get(b, float('inf')))
        matrix.append(row)
    return [[value if value != float('inf') else 10_000_000 for value in row] for row in matrix]

distance_matrix = network_matrix(crashes['network_node'])
results = []
for radius in [250, 500, 1000]:
    labels = DBSCAN(eps=radius, min_samples=3, metric='precomputed').fit(distance_matrix).labels_
    out = crashes[['Crash ID', 'year', 'latitude', 'longitude', 'deaths', 'serious_injuries']].copy()
    out['radius_m'] = radius
    out['cluster'] = labels
    results.append(out)
clusters = pd.concat(results, ignore_index=True)
clusters.to_csv(OUT / 'network_crash_clusters.csv', index=False)
summary = (clusters[clusters['cluster'] >= 0]
           .groupby(['radius_m', 'cluster'], as_index=False)
           .agg(crashes=('Crash ID', 'nunique'), deaths=('deaths', 'sum'),
                serious_injuries=('serious_injuries', 'sum'), first_year=('year', 'min'),
                last_year=('year', 'max')))
summary.to_csv(OUT / 'network_cluster_summary.csv', index=False)
summary.sort_values(['radius_m', 'crashes'], ascending=[True, False])


## 5. Make a simple map

This map shows the 500-meter clusters. It is a reporting aid, not a statistical significance test. Run the table above at all three radii before describing a cluster as robust.

In [ ]:
plot = clusters[clusters['radius_m'] == 500].copy()
plot['label'] = plot['cluster'].where(plot['cluster'] >= 0, -1)
fig, ax = plt.subplots(figsize=(9, 9))
plot.plot(ax=ax, x='longitude', y='latitude', kind='scatter', c='label', cmap='tab10',
          s=28, alpha=.8, colorbar=False)
ax.set_title('San Antonio severe/fatal bicyclist crash clusters
(500-meter network distance, 2016–2025)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
fig.tight_layout()
fig.savefig(OUT / 'network_clusters_500m.png', dpi=200)
plt.show()


## How to write the result

Use cautious language: **“The analysis identified a concentration of X severe or fatal bicyclist crashes within Y meters of one another along the street network.”** Do not call a cluster statistically significant unless we add a randomization test. Also keep 2026 year-to-date separate from the 2016–2025 analysis.